# MERIT: Multi-Source Evaluation Suite
### Imperial College London | Final Year Project

This notebook orchestrates the empirical validation of the MERIT recruitment engine (**Studies 01--15**), matching `run_all_evaluations.py` and the `/evaluation/` directory prefixes.

| Study | Directory |
|------:|-----------|
| 01 | `01-comparative_study` |
| 02 | `02-runtime_study` |
| 03 | `03-spacetime_study` |
| 04 | `04-ir_parser_test` |
| 05 | `05-jd_parser_test` |
| 06 | `06-adversarial_test` |
| 07 | `07-spearman_high_discrimination` |
| 08 | `08-spearman_seniority_bias_audit` |
| 09 | `09-spearman_peer_competition` |
| 10 | `10-spearman_signal_dissonance_failure_case` |
| 11 | `11-shapley_verification` |
| 12 | `12-conflict_resolution` |
| 13 | `13-dynamic_tf_idf_recruiter_validation` |
| 14 | `14-hci_trial` |
| 15 | `15-bias_anonymisation_audit` |


## 0. System Specification
Logging the benchmarking environment to provide context for runtime and spacetime results.

In [ ]:
import platform
import psutil
import sys

def print_sys_info():
    print("="*60)
    print("BENCHMARKING ENVIRONMENT")
    print("="*60)
    print(f"OS: {platform.system()} {platform.release()} ({platform.version()})")
    print(f"Python Version: {sys.version}")
    print(f"Processor: {platform.processor()}")
    print(f"Cores: {psutil.cpu_count(logical=False)} Physical, {psutil.cpu_count(logical=True)} Logical")
    print(f"Total RAM: {psutil.virtual_memory().total / (1024**3):.2f} GB")
    print("="*60)

print_sys_info()

## Setup & Path Integration

In [ ]:
import os
import sys
import subprocess
import contextlib
from IPython.display import Image, display

# Add subdirectories to path to allow direct imports
root_dir = os.getcwd()
sys.path.append(os.path.join(root_dir, '01-comparative_study'))
sys.path.append(os.path.join(root_dir, '02-runtime_study'))
sys.path.append(os.path.join(root_dir, '03-spacetime_study'))

@contextlib.contextmanager
def working_directory(path):
    """Utility to temporarily change the working directory for sub-study runners."""
    prev_cwd = os.getcwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(prev_cwd)

print("Environment Initialised.")

## 1. Study 01: Comparative Evaluation (Adversarial Benchmarking)
This study evaluates the ranking accuracy of MERIT against Traditional and Modern AI ATS models using adversarial personas.

In [ ]:
from baseline_ats import run_baseline_study
from modern_ai_ats import run_ai_study
from merit_cv_only import run_cv_ablation_study
from merit_all_sources import run_full_study
from generate_visualisations import generate_evaluation_visualisations

study_01_dir = os.path.join(root_dir, '01-comparative_study')

with working_directory(study_01_dir):
    print("Running Stage 1: Baseline ATS...")
    run_baseline_study()
    
    print("\nRunning Stage 2: Modern AI ATS...")
    run_ai_study()
    
    print("\nRunning Stage 3: MERIT (CV-Only Ablation)...")
    run_cv_ablation_study()
    
    print("\nRunning Stage 4: MERIT (Full Multi-Source Integration)...")
    run_full_study()
    
    print("\nGenerating Visualisations...")
    generate_evaluation_visualisations()

print("\nVisualising Results:")
display(Image(filename='01-comparative_study/output/rank_displacement_holistic.png', width=800))

## 2. Study 02: Runtime Complexity
Evaluation of ranking latency scaling from $N=10$ to $N=500$ candidates.

In [ ]:
from run_runtime import benchmark_runtime

study_02_dir = os.path.join(root_dir, '02-runtime_study')

with working_directory(study_02_dir):
    print("Running Runtime Complexity Study...")
    benchmark_runtime()
    from generate_runtime_visualisations import generate_runtime_plots
    generate_runtime_plots()

print("\nPerformance Curves:")
display(Image(filename='02-runtime_study/output/runtime_complexity_plot.png', width=800))
display(Image(filename='02-runtime_study/output/runtime_efficiency_log.png', width=800))

## 3. Study 03: Spacetime Complexity
Analysis of static engine footprint and incremental batch memory overhead.

**Methodology:** one fresh Python subprocess per engine (static load) and per (engine, $N$) pair (batch RSS), so measurements are not contaminated by other engines in the same process. The cell below launches `run_spacetime.py` as a child process (same orchestration as `run_all_evaluations.py`). Expect **~10--15 minutes** for the full sweep. To refresh plots only: `python run_spacetime.py --plots-only`.

In [ ]:
study_03_dir = os.path.join(root_dir, '03-spacetime_study')

print("Running Study 03 (isolated subprocess memory benchmark)...")
subprocess.run([sys.executable, 'run_spacetime.py'], cwd=study_03_dir, check=True)

print("\nResource Allocation Profile:")
display(Image(filename='03-spacetime_study/output/spacetime_complexity_composite.png', width=800))

## 4. Study 04: IR Parser Accuracy (Information Retrieval)
This study evaluates the precision, recall, and F1-score of the core CV parser against a 'Ground Truth' dataset of 15 diverse technical resumes.

In [ ]:
study_04_dir = os.path.join(root_dir, '04-ir_parser_test')

with working_directory(study_04_dir):
    print("Running IR Parser Accuracy Test...")
    
    from run_extraction import run_extraction
    run_extraction()
    
    from run_accuracy_test import run_test
    run_test()
    
    from generate_ir_visualisations import generate_accuracy_plots
    generate_accuracy_plots()

print("\nParser Accuracy Metrics:")
display(Image(filename='04-ir_parser_test/output/ir_parser_accuracy.png', width=800))

## 5. Study 05: JD Parser Accuracy
This study evaluates the extraction accuracy for Job Descriptions, focusing on titles, company names, and technical skill requirements.

In [ ]:
study_05_dir = os.path.join(root_dir, '05-jd_parser_test')

with working_directory(study_05_dir):
    print("Running JD Parser Accuracy Test...")

    from run_study import run_study_05
    run_study_05()

print("\nJD Parser Accuracy Metrics:")
display(Image(filename='05-jd_parser_test/output/jd_parser_accuracy.png', width=800))

## 6. Study 06: Adversarial Stress Testing
This study evaluates the robustness of the MERIT recruitment engine against adversarial gaming tactics, including keyword stuffing, identity hijacking (Smart Squatter), and metric inflation.

In [ ]:
print("Running Adversarial Robustness Study...")
with working_directory('06-adversarial_test'):
    # Force reload of run_study to avoid collision with Study 05
    import sys
    import os
    if 'run_study' in sys.modules:
        del sys.modules['run_study']
    
    sys.path.insert(0, os.getcwd())
    from run_study import run_study_06
    run_study_06()

print("\nAdversarial Robustness Profile:")
display(Image(filename='06-adversarial_test/output/adversarial_robustness.png', width=800))

## 7--10. Spearman Rank Correlation Suite
This section validates the engine's ranking accuracy against expert human judgment using Spearman's Rho ($\rho$). We test four distinct scenarios to evaluate performance, bias, and robustness.

### 4.1 Study 07: High Discrimination
Tests the ability to distinguish between Backend specialists and unrelated engineering domains (iOS, Android, Data Science).


In [ ]:
study_path = os.path.join(os.getcwd(), "07-spearman_high_discrimination")
with working_directory(study_path):
    subprocess.run([sys.executable, "run_study.py"], check=True)

display(Image(filename="07-spearman_high_discrimination/output/spearman_chart.png", width=800))


### 4.2 Study 08: Seniority Bias Audit
Tests if engines wrongly prioritize seniority (Principal Engineers) for an Intern-level role. Human ranking prioritizes the Intern.


In [ ]:
study_path = os.path.join(os.getcwd(), "08-spearman_seniority_bias_audit")
with working_directory(study_path):
    subprocess.run([sys.executable, "run_study.py"], check=True)

display(Image(filename="08-spearman_seniority_bias_audit/output/spearman_chart.png", width=800))


### 4.3 Study 09: Peer Competition
Tests technical discrimination among candidates of the same seniority (all Mid-Level) across different engineering specialties.


In [ ]:
study_path = os.path.join(os.getcwd(), "09-spearman_peer_competition")
with working_directory(study_path):
    subprocess.run([sys.executable, "run_study.py"], check=True)

display(Image(filename="09-spearman_peer_competition/output/spearman_chart.png", width=800))


### 4.4 Study 10: Signal Dissonance (Sabotage Case)
A deliberate failure case where GitHub data is sabotaged to contradict the CV. Used to test engine sensitivity to data quality.


In [ ]:
study_path = os.path.join(os.getcwd(), "10-spearman_signal_dissonance_failure_case")
with working_directory(study_path):
    subprocess.run([sys.executable, "run_study.py"], check=True)

display(Image(filename="10-spearman_signal_dissonance_failure_case/output/spearman_chart.png", width=800))


## 11. Study 11: Shapley Value Mathematical Verification
This study verifies the mathematical integrity of the MERIT explainability engine by testing the four Shapley Axioms (Efficiency, Null Player, Symmetry, and Additivity) on a controlled dataset.

In [ ]:
study_11_dir = os.path.join(root_dir, '11-shapley_verification')
with working_directory(study_11_dir):
    # Run the verification script
    !python run_study.py

    # Display the primary verification results
    display(Image(filename=os.path.join('output', 'shapley_attribution_breakdown.png'), width=800))


## 12. Study 12: Bayesian Conflict Resolution Verification
This study provides a worked mathematical walkthrough of the Bayesian Evidence Fusion engine. It demonstrates how the system's prior and posterior beliefs converge when presented with contradictory evidence from multiple sources (e.g., CV vs. GitHub).

In [ ]:
study_12_dir = os.path.join(root_dir, '12-conflict_resolution')
with working_directory(study_12_dir):
    # Run the verification script
    !python run_study.py

    # Display the convergence results
    display(Image(filename=os.path.join('output', 'fusion_convergence.png'), width=800))
    display(Image(filename=os.path.join('output', 'alpha_beta_shift.png'), width=800))

## 13. Study 13: TF-IDF Metric Prediction vs Recruiter Alignment

Pilot comparison of dynamic TF-IDF priority suggestions against recruiter consensus on three held-out job descriptions ($N=4$ raters, $n=27$ skill--role pairs). Requires populated ground truth under `test_data/job_descriptions/`.


In [ ]:
study_13_dir = os.path.join(root_dir, '13-dynamic_tf_idf_recruiter_validation')
print("Running Study 13: TF-IDF vs recruiter priorities...")
with working_directory(study_13_dir):
    subprocess.run([sys.executable, "run_study.py"], check=True)

for chart in [
    'study13_holdout_metrics.png',
    'study13_consensus_vs_tfidf_scatter.png',
    'study13_per_role_dumbbell.png',
    'study13_top_misalignments.png',
    'study13_rater_heatmap_ml.png',
]:
    rel = os.path.join('13-dynamic_tf_idf_recruiter_validation/output', chart)
    if os.path.exists(os.path.join(root_dir, rel)):
        display(Image(filename=rel, width=800))


## 14. Study 14: Human-Computer Interaction (HCI) User Trial and Usability Audit
This study audits the **decision calibration**, **confidence calibration**, and **system usability** of MERIT's interactive "Glass-Box" scorecard under real-world recruiter and technical peer interaction ($N=5$). It evaluates how explainability impacts screening decisions, conviction, and overall user trust.

In [ ]:
study_14_dir = os.path.join(root_dir, '14-hci_trial')
with working_directory(study_14_dir):
    # Run the HCI evaluation script
    !python run_study.py

    # Display the HCI user trial evaluation results
    display(Image(filename=os.path.join('output', 'hci_decision_reversals.png'), width=800))
    display(Image(filename=os.path.join('output', 'hci_confidence_calibration.png'), width=800))
    display(Image(filename=os.path.join('output', 'hci_sus_distribution.png'), width=800))

## 15. Study 15: Bias & Anonymisation Audit (Between-Subjects)

Evaluates whether Blind Mode reduces prestige-driven shortlisting compared to Visible PII. Four recruiters and four peers ($N=8$) were split across conditions ($n=4$ per arm); each evaluator reviewed per-candidate Intelligence Reports only (no cohort ranking overlay). Analysis compares shortlist rates, prestige gap, and MERIT rank alignment between Visible and Blind-default arms.

In [ ]:
study_15_dir = os.path.join(root_dir, '15-bias_anonymisation_audit')

print("Running Study 15: Bias & Anonymisation Audit...")
subprocess.run([sys.executable, 'run_study.py'], cwd=study_15_dir, check=True)

print("\nStudy 15 outputs:")
display(Image(filename='15-bias_anonymisation_audit/output/bias_shortlist_rates.png', width=800))
display(Image(filename='15-bias_anonymisation_audit/output/bias_rank_displacement.png', width=800))
display(Image(filename='15-bias_anonymisation_audit/output/bias_blind_uplift.png', width=800))